### library

In [52]:
import argparse
import json
import math
import os
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.stats import multivariate_normal, kstest
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

### download mnist data

In [30]:
def get_dataloaders():
    tfm = transforms.Compose([transforms.ToTensor()])
    train_set = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
    test_set = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)
    train_loader = DataLoader(train_set, batch_size=128, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_set, batch_size=128, shuffle=False)
    return train_loader, test_loader

### model

In [31]:
class VAE(nn.Module):
    def __init__(self, image_size, latent_dim, hidden_dim):
        super().__init__()
        self.image_size = image_size
        self.input_dim = image_size * image_size
        self.latent_dim = latent_dim

        self.enc_fc1 = nn.Linear(self.input_dim, hidden_dim)
        self.enc_fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)

        self.dec_fc1 = nn.Linear(latent_dim, hidden_dim // 2)
        self.dec_fc2 = nn.Linear(hidden_dim // 2, hidden_dim)
        self.dec_out = nn.Linear(hidden_dim, self.input_dim)

    def encode(self, x):
        h = F.relu(self.enc_fc1(x))
        h = F.relu(self.enc_fc2(h))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        h = F.relu(self.dec_fc2(h))
        return torch.sigmoid(self.dec_out(h))

    def forward(self, x):
        x_flat = x.view(x.size(0), -1)
        mu, logvar = self.encode(x_flat)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

### train

In [32]:
def train_vae(model, train_loader):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = []

    model.train()
    for epoch in range(1, 15 + 1):
        total, total_recon, total_kl, n = 0.0, 0.0, 0.0, 0
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            recon, mu, logvar = model(x)
            loss, recon_loss, kl = vae_loss(recon, x, mu, logvar, 1.0)
            loss.backward()
            optimizer.step()

            total += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl.item()
            n += x.size(0)

        avg_loss, avg_recon, avg_kl = total / n, total_recon / n, total_kl / n
        history.append({"epoch": epoch, "loss": avg_loss, "recon": avg_recon, "kl": avg_kl})
        print(f"[train] epoch {epoch:3d}/{15} | loss {avg_loss:.3f} "
              f"| recon {avg_recon:.3f} | KL {avg_kl:.3f}")

    return history

### evaluate

In [33]:
@torch.no_grad()
def get_latents(model, loader):
    model.eval()
    mus, labels = [], []
    for x, y in loader:
        x = x.to(device)
        x_flat = x.view(x.size(0), -1)
        mu, _ = model.encode(x_flat)
        mus.append(mu.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(mus, axis=0), np.concatenate(labels, axis=0)

### utils-some usefull fuctions

In [34]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

In [35]:
def vae_loss(recon, x, mu, logvar, beta=1.0):
    x_flat = x.view(x.size(0), -1)
    recon_loss = F.binary_cross_entropy(recon, x_flat, reduction="sum")
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_div, recon_loss, kl_div

In [36]:
def class_statistics(z, labels, classes):
    stats = {}
    for c in classes:
        pts = z[labels == c]
        mean = pts.mean(axis=0)
        cov = np.cov(pts.T) + 1e-6 * np.eye(z.shape[1])
        if z.shape[1] == 1:
            cov = np.array([[np.var(pts) + 1e-6]])
        stats[c] = {"mean": mean, "cov": np.atleast_2d(cov), "n": len(pts)}
    return stats

In [37]:
def gaussian_goodness_of_fit(z, labels, stats):
    results = {}
    for c, s in stats.items():
        pts = z[labels == c]
        try:
            rv = multivariate_normal(mean=s["mean"], cov=s["cov"], allow_singular=True)
            mean_ll = float(np.mean(rv.logpdf(pts)))
        except Exception:
            mean_ll = float("nan")

        ks_pvals = []
        for d in range(z.shape[1]):
            m, v = s["mean"][d], s["cov"][d, d]
            std = math.sqrt(max(v, 1e-12))
            stat, p = kstest(pts[:, d], "norm", args=(m, std))
            ks_pvals.append(p)

        results[c] = {"mean_log_likelihood": mean_ll, "ks_pvalues_per_dim": ks_pvals}
    return results

In [38]:
def bhattacharyya_distance(mean1, cov1, mean2, cov2):
    sigma = (cov1 + cov2) / 2.0
    diff = (mean1 - mean2).reshape(-1, 1)
    sigma_inv = np.linalg.inv(sigma)
    term1 = 0.125 * float((diff.T @ sigma_inv @ diff).item())

    det_sigma = np.linalg.det(sigma)
    det1, det2 = np.linalg.det(cov1), np.linalg.det(cov2)
    det1, det2, det_sigma = max(det1, 1e-300), max(det2, 1e-300), max(det_sigma, 1e-300)
    term2 = 0.5 * math.log(det_sigma / math.sqrt(det1 * det2))
    return term1 + term2

In [39]:
def pairwise_distance_matrices(stats, classes):
    n = len(classes)
    euclid = np.zeros((n, n))
    bhatt = np.zeros((n, n))
    for i, ci in enumerate(classes):
        for j, cj in enumerate(classes):
            if i == j:
                continue
            m1, m2 = stats[ci]["mean"], stats[cj]["mean"]
            euclid[i, j] = np.linalg.norm(m1 - m2)
            bhatt[i, j] = bhattacharyya_distance(m1, stats[ci]["cov"], m2, stats[cj]["cov"])
    return euclid, bhatt

In [40]:
def clustering_quality(z, labels):
    try:
        return float(silhouette_score(z, labels))
    except Exception:
        return float("nan")

In [41]:
def reduce_to_2d(z, method="pca"):
    if z.shape[1] == 2:
        return z, None
    if method == "tsne":
        reducer = TSNE(n_components=2, random_state=42, init="pca", perplexity=30)
        z2 = reducer.fit_transform(z)
        return z2, None
    reducer = PCA(n_components=2, random_state=42)
    z2 = reducer.fit_transform(z)
    return z2, reducer

In [42]:
def plot_latent_scatter(z2, labels, classes, out_path, title, ellipses_stats=None):
    fig, ax = plt.subplots(figsize=(8, 7))
    cmap = plt.get_cmap("tab10")
    for c in classes:
        pts = z2[labels == c]
        ax.scatter(pts[:, 0], pts[:, 1], s=6, alpha=0.5, color=cmap(c % 10), label=str(c))

    if ellipses_stats is not None:
        from matplotlib.patches import Ellipse
        for c in classes:
            mean = ellipses_stats[c]["mean"][:2]
            cov = ellipses_stats[c]["cov"][:2, :2]
            vals, vecs = np.linalg.eigh(cov)
            vals = np.maximum(vals, 1e-8)
            angle = math.degrees(math.atan2(vecs[1, 1], vecs[0, 1]))
            width, height = 2 * 2 * np.sqrt(vals)
            ell = Ellipse(mean, width, height, angle=angle, edgecolor=cmap(c % 10),
                           facecolor="none", lw=1.8, zorder=5)
            ax.add_patch(ell)

    ax.set_title(title)
    ax.set_xlabel("dim 1")
    ax.set_ylabel("dim 2")
    ax.legend(title="digit", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

In [43]:
def plot_distance_heatmap(matrix, classes, title, out_path):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(matrix, cmap="viridis")
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f"{matrix[i, j]:.1f}", ha="center", va="center",
                     color="white" if matrix[i, j] < matrix.max() / 2 else "black", fontsize=6)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

In [44]:
def plot_training_curve(history, out_path):
    epochs = [h["epoch"] for h in history]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, [h["loss"] for h in history], label="total loss")
    ax.plot(epochs, [h["recon"] for h in history], label="reconstruction")
    ax.plot(epochs, [h["kl"] for h in history], label="KL divergence")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss per sample")
    ax.set_title("VAE training curve")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

In [45]:
@torch.no_grad()
def plot_latent_manifold(model, out_path, grid_n=15, span=3.0):
    if model.latent_dim != 2:
        return
    model.eval()
    axis = np.linspace(-span, span, grid_n)
    canvas = np.zeros((grid_n * model.image_size, grid_n * model.image_size))
    for i, yi in enumerate(axis):
        for j, xi in enumerate(axis):
            z = torch.tensor([[xi, yi]], dtype=torch.float32, device=device)
            img = model.decode(z).cpu().numpy().reshape(model.image_size, model.image_size)
            canvas[i * model.image_size:(i + 1) * model.image_size,
                   j * model.image_size:(j + 1) * model.image_size] = img

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(canvas, cmap="gray")
    ax.set_title("Decoded images across a 2D grid of the latent space")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

In [46]:
@torch.no_grad()
def plot_class_interpolation(model, stats, class_a, class_b, out_path, steps=10):
    model.eval()
    m1, m2 = stats[class_a]["mean"], stats[class_b]["mean"]
    alphas = np.linspace(0, 1, steps)
    imgs = []
    for a in alphas:
        z = (1 - a) * m1 + a * m2
        z_t = torch.tensor(z, dtype=torch.float32, device=device).unsqueeze(0)
        img = model.decode(z_t).cpu().numpy().reshape(model.image_size, model.image_size)
        imgs.append(img)

    fig, axes = plt.subplots(1, steps, figsize=(steps * 1.2, 1.6))
    for ax, img, a in zip(axes, imgs, alphas):
        ax.imshow(img, cmap="gray")
        ax.axis("off")
        ax.set_title(f"{a:.1f}", fontsize=7)
    fig.suptitle(f"Latent interpolation: digit {class_a} \u2192 digit {class_b}", y=1.15)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

In [47]:
def write_report(classes, stats, fit_results, euclid, bhatt, sil_score, history, out_path):
    lines = []
    lines.append("# Mathematical Analysis of VAE Latent Space (MNIST)\n")
    lines.append(f"- Latent dimensionality: **{2}**")
    lines.append(f"- Training epochs: **{15}**, final loss/sample: "
                 f"**{history[-1]['loss']:.2f}** "
                 f"(reconstruction {history[-1]['recon']:.2f}, KL {history[-1]['kl']:.2f})")
    lines.append(f"- Overall clustering quality (silhouette score, range [-1, 1]): "
                 f"**{sil_score:.3f}**\n")

    lines.append("## 1. Per-class Gaussian fit\n")
    lines.append("| digit | n | mean log-likelihood | min KS p-value across dims |")
    lines.append("|---|---|---|---|")
    for c in classes:
        f = fit_results[c]
        min_p = min(f["ks_pvalues_per_dim"])
        lines.append(f"| {c} | {stats[c]['n']} | {f['mean_log_likelihood']:.2f} | {min_p:.3f} |")
    lines.append("")

    lines.append("## 2. Inter-class separation\n")
    lines.append("Two complementary distance measures were computed between every pair of "
                  "class-conditional Gaussians:\n")
    lines.append("- **Euclidean distance** between class means — a simple geometric measure "
                  "of how far apart class centroids sit in latent space.")

    avg_euclid = euclid[euclid > 0].mean()
    avg_bhatt = bhatt[bhatt > 0].mean()
    most_sep_idx = np.unravel_index(np.argmax(bhatt), bhatt.shape)
    least_sep = bhatt.copy()
    least_sep[least_sep == 0] = np.inf
    least_sep_idx = np.unravel_index(np.argmin(least_sep), least_sep.shape)

    lines.append(f"- Average pairwise Euclidean distance between class means: "
                 f"**{avg_euclid:.2f}**")
    lines.append(f"- Average pairwise Bhattacharyya distance: **{avg_bhatt:.2f}**")
    lines.append(f"- Most separated pair (highest Bhattacharyya distance): "
                 f"digits **{classes[most_sep_idx[0]]}** and **{classes[most_sep_idx[1]]}** "
                 f"(D_B = {bhatt[most_sep_idx]:.2f})")
    lines.append(f"- Most overlapping pair (lowest Bhattacharyya distance): "
                 f"digits **{classes[least_sep_idx[0]]}** and **{classes[least_sep_idx[1]]}** "
                 f"(D_B = {bhatt[least_sep_idx]:.2f})\n")

    lines.append("See `latent_scatter.png` for the visual clustering, "
                 "`distance_euclidean.png` / `distance_bhattacharyya.png` for the full "
                 "pairwise matrices, and `latent_interpolation.png` for a decoded "
                 "traversal between the most-overlapping pair of classes.\n")

    lines.append("## 3. Interpretation\n")

    lines.append("## 4. Files produced\n")
    lines.append("- `training_curve.png` — reconstruction/KL/total loss vs. epoch")
    lines.append("- `latent_scatter.png` — 2D (or reduced) latent space colored by digit, "
                 "with fitted-Gaussian ~2-std contours")
    lines.append("- `distance_euclidean.png`, `distance_bhattacharyya.png` — pairwise class "
                 "distance heatmaps")
    lines.append("- `latent_manifold.png` — decoded images across a 2D latent grid "
                 "(only produced when latent_dim = 2)")
    lines.append("- `latent_interpolation.png` — decoded linear interpolation between two "
                 "digit classes' mean latent vectors")
    lines.append("- `latent_stats.json` — raw numeric results (means, covariances, "
                 "distances, silhouette score)")

    with open(out_path, "w") as f:
        f.write("\n".join(lines))

In [56]:
def main():
    parser = argparse.ArgumentParser(description="VAE latent-space mathematical analysis on MNIST.")
    parser.add_argument("--latent-dim", type=int, default=2)
    parser.add_argument("--hidden-dim", type=int, default=256)
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--beta", type=float, default=1.0)
    parser.add_argument("--reduce", choices=["pca", "tsne"], default="pca",
                        help="Dimensionality reduction used for visualization when latent_dim > 2.")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out-dir", type=str, default="outputs")
    args, unknown = parser.parse_known_args()

    os.makedirs("outputs", exist_ok=True)
    set_seed(42)

    print(f"[config] device={device}, latent_dim={2}, epochs={15}")

    train_loader, test_loader = get_dataloaders()
    model = VAE(28, 2, 256)
    history = train_vae(model, train_loader)
    plot_training_curve(history, os.path.join("outputs", "training_curve.png"))

    z, labels = get_latents(model, test_loader)
    classes = sorted(np.unique(labels).tolist())

    stats = class_statistics(z, labels, classes)
    fit_results = gaussian_goodness_of_fit(z, labels, stats)
    euclid, bhatt = pairwise_distance_matrices(stats, classes)
    sil = clustering_quality(z, labels)

    print(f"[analysis] silhouette score = {sil:.3f}")

    z2, reducer = reduce_to_2d(z, method="pca")
    ellipses = stats
    plot_latent_scatter(
        z2, labels, classes, os.path.join("outputs", "latent_scatter.png"),
        title=f"Latent space (dim={2}"
              f"{', reduced via ' + 'pca'.upper()})",
        ellipses_stats=ellipses,
    )
    plot_distance_heatmap(euclid, classes, "Euclidean distance between class means",
                          os.path.join("outputs", "distance_euclidean.png"))
    plot_distance_heatmap(bhatt, classes, "Bhattacharyya distance between class Gaussians",
                          os.path.join("outputs", "distance_bhattacharyya.png"))

    plot_latent_manifold(model, os.path.join("outputs", "latent_manifold.png"))

    bhatt_masked = bhatt.copy()
    bhatt_masked[bhatt_masked == 0] = np.inf
    i, j = np.unravel_index(np.argmin(bhatt_masked), bhatt_masked.shape)
    class_a, class_b = classes[i], classes[j]
    plot_class_interpolation(model, stats, class_a, class_b,
                             os.path.join("outputs", "latent_interpolation.png"))

    raw_results = {
        "silhouette_score": sil,
        "classes": [int(c) for c in classes],
        "class_stats": {
            str(c): {"mean": [float(x) for x in stats[c]["mean"]],
                     "cov": [[float(x) for x in row] for row in stats[c]["cov"]],
                     "n": int(stats[c]["n"])} for c in classes
        },
        "goodness_of_fit": {
            str(c): {"mean_log_likelihood": float(fit_results[c]["mean_log_likelihood"]),
                     "ks_pvalues_per_dim": [float(p) for p in fit_results[c]["ks_pvalues_per_dim"]]}
            for c in classes
        },
        "euclidean_distance_matrix": [[float(x) for x in row] for row in euclid],
        "bhattacharyya_distance_matrix": [[float(x) for x in row] for row in bhatt],
        "training_history": [{k: float(v) if isinstance(v, float) else int(v) for k, v in h.items()}
                             for h in history],
    }
    with open(os.path.join("outputs", "latent_stats.json"), "w") as f:
        json.dump(raw_results, f, indent=2)

    write_report(classes, stats, fit_results, euclid, bhatt, sil, history,
                 os.path.join("outputs", "outputs/report.md"))

    print(f"[done] All outputs written to: {os.path.abspath('outputs')}")

### run this

In [57]:
main()

[config] device=cpu, latent_dim=2, epochs=15
[train] epoch   1/15 | loss 189.818 | recon 185.987 | KL 3.830
[train] epoch   2/15 | loss 163.676 | recon 158.588 | KL 5.088
[train] epoch   3/15 | loss 158.123 | recon 152.738 | KL 5.384
[train] epoch   4/15 | loss 155.124 | recon 149.583 | KL 5.541
[train] epoch   5/15 | loss 153.078 | recon 147.407 | KL 5.671
[train] epoch   6/15 | loss 151.612 | recon 145.843 | KL 5.769
[train] epoch   7/15 | loss 150.240 | recon 144.344 | KL 5.896
[train] epoch   8/15 | loss 149.099 | recon 143.145 | KL 5.953
[train] epoch   9/15 | loss 148.288 | recon 142.262 | KL 6.026
[train] epoch  10/15 | loss 147.485 | recon 141.418 | KL 6.067
[train] epoch  11/15 | loss 146.856 | recon 140.728 | KL 6.128
[train] epoch  12/15 | loss 146.187 | recon 140.017 | KL 6.170
[train] epoch  13/15 | loss 145.776 | recon 139.567 | KL 6.208
[train] epoch  14/15 | loss 145.056 | recon 138.805 | KL 6.251
[train] epoch  15/15 | loss 144.731 | recon 138.452 | KL 6.279
[analysis]